# 실습 13. 동등성·비열등성 검정

사회약학 연구방법 노트의 '실습 13. 동등성·비열등성 검정'에 딸린 노트북입니다. a 절은 13장 c 절의 HbA1c 시험을 흉내 낸 모의자료로 평균 차이의 비열등성을, b 절은 13장 d 절의 제균 시험 요약자료로 비율 차이의 비열등성을, c 절은 공개 약동학 자료(Theoph)와 13장의 AUC 자료, 교차설계 모의자료로 생물학적 동등성을 계산합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 13에 있습니다.


## a. 평균 차이의 비열등성

이 실습이 쓰는 패키지를 불러와 버전을 확인합니다. c 절의 Theoph 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import scipy, statsmodels, matplotlib

print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scipy       ", scipy.__version__)
print("statsmodels ", statsmodels.__version__)
print("matplotlib  ", matplotlib.__version__)

### 셀 2. 모의 임상시험 자료 만들기

In [ ]:
rng = np.random.default_rng(2024)    # 시드를 고정해 같은 난수 재현
new = rng.normal(loc=-0.74, scale=1.02, size=185)   # 신약 X
ref = rng.normal(loc=-0.83, scale=1.06, size=183)   # 대조약
print(new[:5].round(2))
for name, x in [("drug X", new), ("comparator", ref)]:
    print(f"{name}: n={x.size}, mean={x.mean():.3f}, "
          f"SD={x.std(ddof=1):.3f}")

### 셀 3. 차이와 95% 신뢰구간

In [ ]:
res = stats.ttest_ind(new, ref, equal_var=False)   # Welch t 검정
d = new.mean() - ref.mean()                        # 신약 - 대조약
ci = res.confidence_interval(confidence_level=0.95)
print(f"difference = {d:.3f}")
print(f"95% CI {ci.low:.3f} to {ci.high:.3f}")
print(f"P (difference = 0) = {res.pvalue:.3f}")

### 셀 4. 비열등성 검정 (단측)

In [ ]:
margin = 0.4
ni = stats.ttest_ind(new - margin, ref, equal_var=False,
                     alternative="less")
print(ni)
se = np.sqrt(new.var(ddof=1) / new.size + ref.var(ddof=1) / ref.size)
print(f"SE = {se:.4f}, (d - margin) / SE = {(d - margin) / se:.3f}")
ci1 = ni.confidence_interval(confidence_level=0.975)   # 단측 97.5% 구간
print(f"one-sided 97.5% CI: {ci1.low} to {ci1.high + margin:.3f}")

### 셀 5. 신뢰구간과 한계를 한 그림에

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.errorbar(d, 0, xerr=[[d - ci.low], [ci.high - d]], fmt="s",
            color="C0", capsize=6, label="Difference (95% CI)")
ax.axvline(0, color="black", lw=1)
ax.axvline(margin, color="C1", ls="--", label="+margin (0.4)")
ax.axvline(-margin, color="grey", ls=":", label="-margin (-0.4)")
ax.set_xlim(-0.6, 0.6)
ax.set_yticks([])
ax.set_xlabel("HbA1c change, drug X minus comparator (%-points)\n"
              "<- favors drug X          favors comparator ->")
ax.legend(loc="upper left", fontsize=8)
plt.show()

### 셀 6. 요약 통계로 13장 b·c 절의 숫자 재현

In [ ]:
def ni_from_stats(label, m1, s1, n1, m2, s2, n2, margin, higher_better):
    v1, v2 = s1**2 / n1, s2**2 / n2               # 두 평균의 분산(SE의 제곱)
    se = np.sqrt(v1 + v2)                          # 차이의 표준오차
    df = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))  # Welch 자유도
    tc = stats.t.ppf(0.975, df)                    # 가운데 95%의 임계값
    d = m1 - m2                                    # 신약 - 대조
    if higher_better:      # 한계 -margin: 신약에 margin을 더하고 "greater"
        r = stats.ttest_ind_from_stats(m1 + margin, s1, n1, m2, s2, n2,
                                       equal_var=False, alternative="greater")
    else:                  # 한계 +margin: 신약에서 margin을 빼고 "less"
        r = stats.ttest_ind_from_stats(m1 - margin, s1, n1, m2, s2, n2,
                                       equal_var=False, alternative="less")
    print(f"{label}: difference {d:.2f}, SE {se:.4f}, df {df:.1f},"
          f" 95% CI {d - tc * se:.2f} to {d + tc * se:.2f},"
          f" t {r.statistic:.2f}, one-sided P {r.pvalue:.4f}")

ni_from_stats("PDC (ch13 b)", 83.9, 16.8, 304, 85.2, 17.5, 304, 5, True)
ni_from_stats("HbA1c (ch13 c)", -0.74, 1.02, 185, -0.83, 1.06, 183, 0.4, False)

## b. 비율 차이의 비열등성

### 셀 7. 위험차의 Wald 구간과 Newcombe 구간

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

trials = {"main": (186, 224, 190, 224),     # 성공 수, 인원 (신약, 표준)
          "small": (48, 50, 50, 50)}
rows = []
for name, (x1, n1, x2, n2) in trials.items():
    for m in ["wald", "newcomb"]:
        lo, hi = confint_proportions_2indep(x1, n1, x2, n2,
                                            method=m, compare="diff")
        rows.append([name, m, x1 / n1 - x2 / n2, lo, hi])
tab = pd.DataFrame(rows, columns=["trial", "method", "diff",
                                  "lower", "upper"])
tab["NI (lower > -0.10)"] = tab["lower"] > -0.10
tab.round(4)

### 셀 8. 네 신뢰구간 그리기

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
for i, r in tab.iterrows():
    ax.errorbar(r["diff"] * 100, -i, capsize=4, fmt="s",
                xerr=[[(r["diff"] - r["lower"]) * 100],
                      [(r["upper"] - r["diff"]) * 100]],
                color="C0" if r["method"] == "wald" else "C1")
ax.axvline(0, color="black", lw=1)
ax.axvline(-10, color="grey", ls=":")
ax.set_yticks(-tab.index, tab["trial"] + ", " + tab["method"])
ax.set_xlabel("Difference in eradication rate, new minus standard "
              "(%-points)")
plt.show()

score 방법이 실패하는 것을 보여 주는 셀(셀 9)은 일부러 오류를 내므로 이 노트북에서 뺐습니다(사이트의 실습 13 b 절에 있습니다).

### 셀 10. statsmodels의 score(Farrington–Manning) 검정

In [ ]:
from statsmodels.stats.proportion import test_proportions_2indep

fm = test_proportions_2indep(186, 224, 190, 224, value=-0.10,
                             method="score", compare="diff",
                             alternative="larger", correction=False)
print(f"z = {fm.statistic:.3f}, one-sided P = {fm.pvalue:.4f}")

### 셀 11. Farrington–Manning 검정을 직접 계산

In [ ]:
from scipy.optimize import minimize_scalar

x1, n1, x2, n2, delta = 186, 224, 190, 224, -0.10

def negloglik(p2):          # 귀무가설 p1 - p2 = delta 아래의 -로그우도
    p1 = p2 + delta
    return -(stats.binom.logpmf(x1, n1, p1)
             + stats.binom.logpmf(x2, n2, p2))

opt = minimize_scalar(negloglik, bounds=(0.1001, 0.9999),
                      method="bounded")
p2t = opt.x
p1t = p2t + delta
se0 = np.sqrt(p1t * (1 - p1t) / n1 + p2t * (1 - p2t) / n2)
z = (x1 / n1 - x2 / n2 - delta) / se0
print(f"restricted MLE: new {p1t:.4f}, standard {p2t:.4f}")
print(f"z = {z:.3f}, one-sided P = {stats.norm.sf(z):.4f}")

## c. 생물학적 동등성

### 셀 12. Theoph 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/csv/"
       "datasets/Theoph.csv")
theo = pd.read_csv(url).drop(columns="rownames")
print(theo.shape, theo["Subject"].nunique())
theo.head(11)

### 셀 13. 혈중농도-시간 곡선

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for sid, g in theo.groupby("Subject"):
    ax.plot(g["Time"], g["conc"], marker="o", ms=3, alpha=0.7)
ax.set_xlabel("Time since dose (h)")
ax.set_ylabel("Theophylline concentration (mg/L)")
plt.show()

### 셀 14. 사다리꼴 규칙으로 AUC 계산

In [ ]:
from scipy.integrate import trapezoid

rows = []
for sid, g in theo.groupby("Subject"):
    g = g.sort_values("Time")
    rows.append({"Subject": sid,
                 "AUC_0_t": trapezoid(g["conc"], g["Time"]),
                 "Cmax": g["conc"].max(),
                 "Tmax": g.loc[g["conc"].idxmax(), "Time"]})
pk = pd.DataFrame(rows)
pk.round(2)

### 셀 15. 기하평균비와 90% 신뢰구간 (대응 자료)

In [ ]:
auc_R = np.array([412, 538, 365, 620, 488, 297,
                  554, 431, 702, 389, 515, 460])   # 대조약
auc_T = np.array([366, 455, 303, 548, 556, 268,
                  535, 510, 639, 429, 412, 525])   # 시험약
res = stats.ttest_rel(np.log(auc_T), np.log(auc_R))
ci90 = res.confidence_interval(confidence_level=0.90)
gmr = np.exp(np.mean(np.log(auc_T) - np.log(auc_R)))
print(f"GMR = {gmr * 100:.2f}%")
print(f"90% CI {np.exp(ci90.low) * 100:.2f}% "
      f"to {np.exp(ci90.high) * 100:.2f}%")

### 셀 16. TOST: 두 번의 단측 검정

In [ ]:
dl = np.log(auc_T) - np.log(auc_R)          # 대상자별 ln(T/R)
lower = stats.ttest_1samp(dl, np.log(0.80), alternative="greater")
upper = stats.ttest_1samp(dl, np.log(1.25), alternative="less")
print(f"t_L = {lower.statistic:.2f}, P = {lower.pvalue:.5f}")
print(f"t_U = {upper.statistic:.2f}, P = {upper.pvalue:.5f}")
print(f"paired t test of no difference: P = {res.pvalue:.2f}")

### 셀 17. 2×2 교차설계 모의자료

In [ ]:
rng = np.random.default_rng(13)
rows = []
for i in range(1, 25):                      # 대상자 24명
    seq = "TR" if i <= 12 else "RT"         # 순서군
    u = rng.normal(0, 0.25)                 # 개인의 흡수 수준
    for period in (1, 2):
        form = seq[period - 1]              # 이번 기간의 제제
        mu = (np.log(4500) + u + 0.02 * (period == 2)
              + np.log(0.97) * (form == "T"))
        rows.append([i, seq, period, form,
                     np.exp(mu + rng.normal(0, 0.13))])
be = pd.DataFrame(rows, columns=["subject", "seq", "period",
                                 "form", "auc"])
be.head(4)

### 셀 18. 교차설계 분산분석으로 90% 신뢰구간

In [ ]:
import statsmodels.formula.api as smf

fit = smf.ols("np.log(auc) ~ C(subject) + C(period) + C(form)",
              data=be).fit()
t = "C(form)[T.T]"
lo, hi = fit.conf_int(alpha=0.10).loc[t]       # 90% 신뢰구간
print(f"GMR = {np.exp(fit.params[t]) * 100:.2f}%, "
      f"90% CI {np.exp(lo) * 100:.2f}% to {np.exp(hi) * 100:.2f}%")
print(f"residual df = {fit.df_resid:.0f}, MSE = {fit.mse_resid:.4f}")
print(f"CV_intra = {np.sqrt(np.exp(fit.mse_resid) - 1) * 100:.1f}%")

## 과제 정답

**과제 1.** 셀 18의 결과(GMR 95.82%, 90% CI 89.71–102.36%)를 논문의 Results처럼 한 문장으로 옮기고, 이 결과가 말해 주지 않는 것을 두 가지 이상 쓰세요. 대상자마다 시험약 AUC ÷ 대조약 AUC를 구해 80–125% 밖인 사람이 몇 명인지도 확인하세요.

### 셀 19. 과제 1 정답. 기하평균비의 90% 신뢰구간 읽기

In [ ]:
wide = be.pivot(index="subject", columns="form", values="auc")  # 한 사람 = 한 행
ratio = wide["T"] / wide["R"]                     # 대상자별 T/R
print(wide.head(3).round(0))
print("geometric mean of T/R:",
      round(np.exp(np.log(ratio).mean()) * 100, 2), "%")
print("range of T/R:", round(ratio.min(), 3), "to", round(ratio.max(), 3))
outside = (ratio < 0.80) | (ratio > 1.25)
print("outside 80-125%:", outside.sum(), "of", len(ratio))

**과제 2.** 셀 7의 본 시험(186/224 대 190/224)을 위험차 대신 비의 척도로 다시 판정하세요. `confint_proportions_2indep`에서 `compare="ratio"`, `method="log"`로 바꿔 제균 성공의 비와 실패의 비의 95% 신뢰구간을 구하고, 설계 때 가정한 제균율 85%에서 −10%p 한계가 각각 어떤 비의 한계가 되는지 계산해 결론을 비교하세요.

### 셀 20. 과제 2 정답. 위험차 대신 비의 척도로 판정하기

In [ ]:
counts = {"success": (186, 224, 190, 224),     # 제균 성공 수, 인원 (신약, 표준)
          "failure": (38, 224, 34, 224)}       # 실패 수 = 224 - 성공 수
margins = {"success": 0.75 / 0.85,             # 85%가 75%로: 성공의 비 한계
           "failure": 0.25 / 0.15}             # 15%가 25%로: 실패의 비 한계
for name, (x1, n1, x2, n2) in counts.items():
    lo, hi = confint_proportions_2indep(x1, n1, x2, n2,
                                        method="log", compare="ratio")
    se_log = np.sqrt(1 / x1 - 1 / n1 + 1 / x2 - 1 / n2)   # ln(비)의 SE
    print(f"{name}: ratio {x1 / n1 / (x2 / n2):.3f},"
          f" 95% CI {lo:.3f} to {hi:.3f}, margin {margins[name]:.3f},"
          f" SE(log) {se_log:.3f}")

**과제 3.** 동료가 소규모 시험(48/50 대 50/50)을 이렇게 요약했습니다. "Fisher 정확검정에서 두 요법의 제균율 차이는 유의하지 않았다(P = 0.49). 위험차의 95% CI는 −9.4 ~ 1.4%p(Wald)로 하한이 한계 −10%p보다 크므로 새 요법은 표준 요법에 비열등하다. 따라서 두 요법의 효과는 같다." 이 결론에서 틀린 곳 세 가지를 찾아 왜 틀렸는지 설명하고, 바르게 고쳐 쓰세요.

### 셀 21. 과제 3 정답. 소규모 시험을 '차이 없음'과 Wald 구간으로 판단한 분석

In [ ]:
print("Fisher exact P (difference = 0):",
      round(stats.fisher_exact([[48, 2], [50, 0]]).pvalue, 3))
print(tab[tab["trial"] == "small"].round(4))      # 셀의 Wald와 Newcombe 구간
for m in ["wald", "score"]:
    r = test_proportions_2indep(48, 50, 50, 50, value=-0.10, method=m,
                                compare="diff", alternative="larger",
                                correction=False)
    print(f"NI test, {m:5}: z = {r.statistic:.3f},"
          f" one-sided P = {r.pvalue:.4f}")